In [29]:
import os
if not os.path.isdir('nanoVLM'):
    !git clone https://github.com/huggingface/nanoVLM.git

!pip -q install minigrid gymnasium
!pip -q install torch torchvision
!pip -q install datasets transformers huggingface_hub pillow matplotlib tqdm

import sys
sys.path.insert(0, 'nanoVLM')

In [30]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/nanoVLM-MiniGrid'
import os
os.makedirs(f'{SAVE_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{SAVE_DIR}/plots', exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [31]:
import random, copy, math, time, re
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from collections import defaultdict, deque
from tqdm import tqdm
from dataclasses import dataclass, field

import gymnasium as gym
import minigrid
from minigrid.wrappers import RGBImgObsWrapper

from models.vision_language_model import VisionLanguageModel
from data.processors import get_image_processor, get_tokenizer


import os
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

os.makedirs("results/plots", exist_ok=True)
os.makedirs("results/checkpoints", exist_ok=True)

SFT_EPOCHS = 6
SFT_BS = 4
SFT_EVAL_EVERY = 5
SFT_EVAL_N = 5

GRPO_ITERS = 20
GRPO_GROUP = 3
GRPO_EVAL_EVERY = 19
GRPO_EVAL_N = 5

FINAL_EVAL_N = 10

print(f"SFT: {SFT_EPOCHS} ep, bs={SFT_BS}, eval every {SFT_EVAL_EVERY}, eval_n={SFT_EVAL_N}")
print(f"GRPO: {GRPO_ITERS} iters, group={GRPO_GROUP}, eval_n={GRPO_EVAL_N}")


ACTION_NAMES = {0: "left", 1: "right", 2: "forward"}
NAME_TO_ACTION = {"left": 0, "right": 1, "forward": 2}

TRAIN_ENV = 'MiniGrid-Empty-8x8-v0'
TEST_ENVS = [
    'MiniGrid-Empty-6x6-v0',
    'MiniGrid-Empty-8x8-v0',
    'MiniGrid-Empty-16x16-v0',
]

Device: cuda
SFT: 6 ep, bs=4, eval every 5, eval_n=5
GRPO: 20 iters, group=3, eval_n=5


In [32]:
from dataclasses import dataclass, field

import transformers
from transformers import LlamaConfig

if not getattr(LlamaConfig, '_rope_patched', False):
    _orig_getattr = getattr(LlamaConfig, '__getattr__', None)

    def _safe_getattr(self, key):
        if key == 'rope_theta':
            return 100000
        if _orig_getattr is not None and _orig_getattr is not _safe_getattr:
            return _orig_getattr(self, key)
        raise AttributeError(f"'{type(self).__name__}' has no attribute '{key}'")

    LlamaConfig.__getattr__ = _safe_getattr
    LlamaConfig._rope_patched = True  # флаг: уже патчили
    print("✓ Patched LlamaConfig (rope_theta fallback = 100000)")
else:
    print("✓ LlamaConfig already patched")

_t = LlamaConfig()
print(f"  rope_theta = {_t.rope_theta}")
del _t

@dataclass
class VLMConfig:
    vit_hidden_dim: int = 768
    vit_inter_dim: int = 4 * vit_hidden_dim
    vit_patch_size: int = 16
    vit_img_size: int = 512
    vit_n_heads: int = 12
    vit_dropout: float = 0.0
    vit_n_blocks: int = 12
    vit_ln_eps: float = 1e-6
    vit_cls_flag: bool = False
    vit_model_type: str = 'google/siglip2-base-patch16-512'

    lm_hidden_dim: int = 960
    lm_inter_dim: int = 2560
    lm_rms_eps: float = 1e-5
    lm_re_base: int = 100000
    lm_max_position_embeddings: int = 8192
    lm_base_vocab_size: int = 49152
    extra_token_amount: int = 66
    lm_vocab_size: int = lm_base_vocab_size + extra_token_amount
    lm_n_heads: int = 15
    lm_n_kv_heads: int = 5
    lm_dropout: float = 0.0
    lm_n_blocks: int = 32
    lm_attn_scaling: float = 1.0
    lm_max_length: int = 192
    lm_use_tokens: bool = False
    lm_tie_weights: bool = True
    lm_model_type: str = 'HuggingFaceTB/SmolLM2-135M'
    lm_tokenizer: str = 'HuggingFaceTB/SmolLM2-360M-Instruct'
    lm_chat_template: str = (
        "{% for message in messages %}"
        "{{'<|im_start|>' + message['role'] + '\\n' + message['content'] + '<|im_end|>' + '\\n'}}"
        "{% endfor %}"
        "{% if add_generation_prompt %}{{ '<|im_start|>assistant\\n' }}{% endif %}"
    )

    mp_pixel_shuffle_factor: int = 4
    mp_image_token_length: int = 64
    max_img_size: int = 512
    resize_to_max_side_len: bool = False

    vlm_extra_tokens: dict = field(default_factory=lambda: {
        "image_token": "<|image|>",
        "global_image_token": "<|global_image|>",
        **{f"r{r}c{c}": f"<row_{r}_col_{c}>"
           for r in range(1, 9) for c in range(1, 9)}
    })
    vlm_load_backbone_weights: bool = True
    vlm_checkpoint_path: str = 'checkpoints'
    hf_repo_name: str = 'nanoVLM'


vlm_cfg = VLMConfig()

print(f"\nVLM Config:")
print(f"  Vision: {vlm_cfg.vit_model_type} ({vlm_cfg.vit_img_size}px)")
print(f"  LM: {vlm_cfg.lm_model_type} (h={vlm_cfg.lm_hidden_dim}, blocks={vlm_cfg.lm_n_blocks})")
print(f"  Image tokens: {vlm_cfg.mp_image_token_length}")
print(f"  Max length: {vlm_cfg.lm_max_length}")

model = VisionLanguageModel(vlm_cfg, load_backbone=vlm_cfg.vlm_load_backbone_weights)
model = model.to(device)

n_total = sum(p.numel() for p in model.parameters())
n_vision = sum(p.numel() for p in model.vision_encoder.parameters())
n_mp = sum(p.numel() for p in model.MP.parameters())
n_decoder = sum(p.numel() for p in model.decoder.parameters())
print(f"  Total params:    {n_total:>12,}")
print(f"  Vision encoder:  {n_vision:>12,}")
print(f"  Modality Proj:   {n_mp:>12,}")
print(f"  LM Decoder:      {n_decoder:>12,}")


image_processor = get_image_processor(vlm_cfg.max_img_size, vlm_cfg.vit_img_size,
                                       vlm_cfg.resize_to_max_side_len)
tokenizer = get_tokenizer(vlm_cfg.lm_tokenizer, vlm_cfg.vlm_extra_tokens,
                           vlm_cfg.lm_chat_template)

print(f"\n  Tokenizer vocab: {len(tokenizer)}")
print(f"  '<|image|>' → id {tokenizer.convert_tokens_to_ids('<|image|>')}")
print(f"  EOS: '{tokenizer.eos_token}' → id {tokenizer.eos_token_id}")
print(f"  PAD: '{tokenizer.pad_token}' → id {tokenizer.pad_token_id}")

_test_img = Image.new('RGB', (56, 56), color=(100, 150, 200))
_test_pv = image_processor(_test_img)

print(f"  image_processor returns: {type(_test_pv)}")
if isinstance(_test_pv, tuple):
    print(f"  Tuple length: {len(_test_pv)}")
    for i, item in enumerate(_test_pv):
        if hasattr(item, 'shape'):
            print(f"    [{i}] tensor shape: {item.shape}")
        else:
            print(f"    [{i}] type: {type(item)}, value: {item}")
    _test_pv = _test_pv[0]
print(f"  Image tensor shape: {_test_pv.shape}")

_img_tokens = vlm_cfg.vlm_extra_tokens["image_token"] * vlm_cfg.mp_image_token_length
_messages = [{"role": "user", "content": f"{_img_tokens}What do you see?"}]
_text = tokenizer.apply_chat_template(_messages, add_generation_prompt=True, tokenize=False)
_test_ids = tokenizer(_text, return_tensors="pt", truncation=True,
                       max_length=vlm_cfg.lm_max_length).input_ids.to(device)
print(f"  Input ids shape: {_test_ids.shape}")

with torch.no_grad():
    _logits, _loss = model(_test_ids, [_test_pv])
print(f"  Logits shape: {_logits.shape}")

_next_token = _logits[:, -1, :].argmax(-1)
print(f"  First generated token: '{tokenizer.decode(_next_token.item())}' (id={_next_token.item()})")

del _test_pv, _test_ids, _logits, _next_token
torch.cuda.empty_cache()

print("\n✓ NanoVLM loaded and verified!")

✓ LlamaConfig already patched
  rope_theta = 100000

VLM Config:
  Vision: google/siglip2-base-patch16-512 (512px)
  LM: HuggingFaceTB/SmolLM2-135M (h=960, blocks=32)
  Image tokens: 64
  Max length: 192
Loading from backbone weights
Successfully loaded google/siglip2-base-patch16-512 weights from safetensors. Model has 86,433,024 parameters.
Extending token embeddings from torch.Size([49152, 576]) to torch.Size([49218, 576])
Initialized 66 new token embeddings
Successfully loaded HuggingFaceTB/SmolLM2-135M weights from safetensors. Model has 134,553,024 parameters.
  Total params:     228,063,936
  Vision encoder:    86,433,024
  Modality Proj:      7,077,888
  LM Decoder:       134,553,024
Resize to max side len: False

  Tokenizer vocab: 49218
  '<|image|>' → id 49152
  EOS: '<|im_end|>' → id 2
  PAD: '<|im_end|>' → id 2
  image_processor returns: <class 'tuple'>
  Tuple length: 2
    [0] tensor shape: torch.Size([1, 3, 512, 512])
    [1] type: <class 'tuple'>, value: (1, 1)
  Image

In [33]:
def make_env(env_id='MiniGrid-Empty-8x8-v0', seed=None):
    """MiniGrid с RGB partial observation (7×7 тайлов → RGB изображение).

    RGBImgObsWrapper конвертирует символическое 7×7×3 наблюдение
    в RGB-изображение размером (tile_size*7, tile_size*7, 3).
    """
    env = gym.make(env_id, render_mode='rgb_array')
    env = RGBImgObsWrapper(env)
    if seed is not None:
        env.reset(seed=seed)
    return env


def obs_to_pil(obs):
    """Наблюдение MiniGrid → PIL Image."""
    img = obs['image'] if isinstance(obs, dict) else obs
    return Image.fromarray(img.astype(np.uint8))


class BFSExpert:
    """BFS shortest-path planner в пространстве (x, y, direction).

    Почему BFS:
    ─────────────
    В MiniGrid агент имеет 3 действия: left (повернуть), right (повернуть),
    forward (шаг вперёд). Поворот — полноценное действие, занимающее один такт.

    Наивная Manhattan-эвристика (идти к цели напрямую) не учитывает повороты
    и может давать неоптимальные траектории. Например, если агент смотрит
    в противоположную сторону от цели, нужно 2 поворота — эвристика это игнорирует.

    BFS в 3D-пространстве (x, y, direction) гарантирует кратчайший путь
    по числу ДЕЙСТВИЙ (не по евклидову расстоянию). В EmptyEnv (без стен)
    BFS завершается мгновенно — граф маленький.

    Эксперт использует полное знание карты (cheating) — знает позицию цели.
    Агент видит только partial obs 7×7 — это создаёт distribution shift,
    особенно на 16×16 где цель часто за пределами видимости.

    Действия:
    ─────────
    0 = left
    1 = right
    2 = forward

    Направления:
    ────────────────────────
    0 = вправо (+x), 1 = вниз (+y), 2 = влево (-x), 3 = вверх (-y)
    """
    DIR_VEC = {0: (1, 0), 1: (0, 1), 2: (-1, 0), 3: (0, -1)}

    def __init__(self, env):
        self.env = env

    def get_action(self):
        uw = self.env.unwrapped
        grid = uw.grid
        pos = tuple(uw.agent_pos)
        d = uw.agent_dir

        goal = None
        for x in range(grid.width):
            for y in range(grid.height):
                c = grid.get(x, y)
                if c and c.type == 'goal':
                    goal = (x, y)
        if goal is None or pos == goal:
            return 2

        start = (pos[0], pos[1], d)
        queue = deque([(start, [])])
        visited = {start}

        while queue:
            (x, y, dr), actions = queue.popleft()
            for action in [0, 1, 2]:
                nx, ny, nd = x, y, dr
                if action == 0:
                    nd = (dr - 1) % 4
                elif action == 1:
                    nd = (dr + 1) % 4
                else:
                    dx, dy = self.DIR_VEC[dr]
                    nx, ny = x + dx, y + dy

                    if not (0 <= nx < grid.width and 0 <= ny < grid.height):
                        continue

                    cell = grid.get(nx, ny)
                    if cell is not None and cell.type == 'wall':
                        continue

                state = (nx, ny, nd)
                if state in visited:
                    continue
                visited.add(state)
                new_actions = actions + [action]

                if (nx, ny) == goal:
                    return new_actions[0]

                queue.append((state, new_actions))

        return 2


env_test = make_env('MiniGrid-Empty-8x8-v0')
obs_test, info_test = env_test.reset(seed=42)
print(f"Observation type: {type(obs_test)}")
print(f"Image shape: {obs_test['image'].shape}")
print(f"Image dtype: {obs_test['image'].dtype}")
print(f"Image range: [{obs_test['image'].min()}, {obs_test['image'].max()}]")
print(f"Agent pos: {env_test.unwrapped.agent_pos}, dir: {env_test.unwrapped.agent_dir}")

expert_test = BFSExpert(env_test)
action = expert_test.get_action()
print(f"Expert action: {action} ({ACTION_NAMES[action]})")

obs, _ = env_test.reset(seed=42)
expert = BFSExpert(env_test)
steps = 0
for s in range(200):
    a = expert.get_action()
    obs, r, term, trunc, _ = env_test.step(a)
    steps += 1
    if term or trunc:
        break
print(f"Expert episode: {steps} steps, reward={r:.3f}, terminated={term}")

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

env_vis = make_env('MiniGrid-Empty-8x8-v0')
obs_vis, _ = env_vis.reset(seed=42)
axes[0].imshow(obs_vis['image'])
axes[0].set_title("Partial Obs (agent view)")
axes[0].axis('off')

full_img = env_vis.unwrapped.get_frame(highlight=True, tile_size=32)
axes[1].imshow(full_img)
axes[1].set_title("Full Grid (8×8)")
axes[1].axis('off')

pil_test = obs_to_pil(obs_vis)
axes[2].imshow(pil_test)
axes[2].set_title(f"PIL Image ({pil_test.size[0]}×{pil_test.size[1]})")
axes[2].axis('off')

plt.tight_layout()
plt.savefig("results/plots/env_overview.png", dpi=150, bbox_inches='tight')
plt.show()

print("\nExpert statistics (10 episodes each):")
for eid in TEST_ENVS:
    steps_list, success = [], 0
    for ep in range(10):
        env = make_env(eid)
        obs, _ = env.reset(seed=ep * 100)
        expert = BFSExpert(env)
        for s in range(500):
            a = expert.get_action()
            obs, r, term, trunc, _ = env.step(a)
            if term or trunc:
                break
        steps_list.append(s + 1)
        if term and r > 0:
            success += 1
        env.close()
    tag = eid.split('-')[2] + '-' + eid.split('-')[3]
    print(f"  {tag}: SR={success}/10, "
          f"steps={np.mean(steps_list):.1f}±{np.std(steps_list):.1f}")

env_test.close()
env_vis.close()
print("\n✓ Environment and BFS expert verified")

Observation type: <class 'dict'>
Image shape: (64, 64, 3)
Image dtype: uint8
Image range: [0, 255]
Agent pos: (1, 1), dir: 0
Expert action: 2 (forward)
Expert episode: 11 steps, reward=0.961, terminated=True

Expert statistics (10 episodes each):
  6x6-v0: SR=10/10, steps=7.0±0.0
  8x8-v0: SR=10/10, steps=11.0±0.0
  16x16-v0: SR=10/10, steps=27.0±0.0

✓ Environment and BFS expert verified


In [34]:
### Cell 4: Сбор данных + ключевые функции

def obs_to_pil(obs_or_env):
    """Конвертирует render() в PIL."""
    if isinstance(obs_or_env, np.ndarray):
        return Image.fromarray(obs_or_env.astype(np.uint8))
    if isinstance(obs_or_env, Image.Image):
        return obs_or_env
    raise ValueError(f"Cannot convert: {type(obs_or_env)}")

def get_optimal_action(env):
    """BFS для получения оптимального действия."""
    agent_pos = env.unwrapped.agent_pos
    agent_dir = env.unwrapped.agent_dir
    goal_pos = None
    grid = env.unwrapped.grid

    for x in range(grid.width):
        for y in range(grid.height):
            cell = grid.get(x, y)
            if cell is not None and cell.type == 'goal':
                goal_pos = (x, y)
                break

    if goal_pos is None:
        return 2  # forward

    DIR_VEC = [(1,0),(0,1),(-1,0),(0,-1)]
    from collections import deque
    queue = deque()
    queue.append((agent_pos[0], agent_pos[1], agent_dir, []))
    visited = set()
    visited.add((agent_pos[0], agent_pos[1], agent_dir))

    while queue:
        x, y, d, actions = queue.popleft()
        if (x, y) == tuple(goal_pos):
            return actions[0] if actions else 2

        # forward
        dx, dy = DIR_VEC[d]
        nx, ny = x + dx, y + dy
        if 0 <= nx < grid.width and 0 <= ny < grid.height:
            cell = grid.get(nx, ny)
            if cell is None or cell.type == 'goal':
                state = (nx, ny, d)
                if state not in visited:
                    visited.add(state)
                    queue.append((nx, ny, d, actions + [2]))

        # left
        nd = (d - 1) % 4
        state = (x, y, nd)
        if state not in visited:
            visited.add(state)
            queue.append((x, y, nd, actions + [0]))

        # right
        nd = (d + 1) % 4
        state = (x, y, nd)
        if state not in visited:
            visited.add(state)
            queue.append((x, y, nd, actions + [1]))

    return 2

ACTION_NAMES = ["left", "right", "forward"]
NAME_TO_ACTION = {"left": 0, "right": 1, "forward": 2}

def collect_expert(env_id, n_episodes, max_steps=50, seed_start=0):
    data = []
    for ep in range(n_episodes):
        env = make_env(env_id)
        obs, _ = env.reset(seed=seed_start + ep)
        for step in range(max_steps):
            pil = obs_to_pil(env.render())
            action = get_optimal_action(env)
            data.append((pil, ACTION_NAMES[action]))
            obs, r, term, trunc, _ = env.step(action)
            if term or trunc:
                break
        env.close()
    return data

print("Collecting expert data...")
train_data = collect_expert('MiniGrid-Empty-8x8-v0', 40, seed_start=0)
train_data += collect_expert('MiniGrid-Empty-6x6-v0', 20, seed_start=1000)
val_data = collect_expert('MiniGrid-Empty-8x8-v0', 8, seed_start=5000)

random.shuffle(train_data)
print(f"Train: {len(train_data)}, Val: {len(val_data)}")
counts = {}
for _, a in train_data:
    counts[a] = counts.get(a, 0) + 1
print(f"Action dist: {counts}")

Train: 580, Val: 88
Action dist: {'right': 60, 'forward': 520}


In [35]:
def process_image(pil_img):
    result = image_processor(pil_img)
    if isinstance(result, tuple):
        return result[0]
    return result

In [36]:
PROMPT_ACTION = (
    "You control an agent in a grid world. "
    "It must reach the green goal. "
    "What action? Answer one word: left, right, or forward."
)

_img_str = vlm_cfg.vlm_extra_tokens["image_token"] * vlm_cfg.mp_image_token_length

_msgs_prompt = [{"role": "user", "content": f"{_img_str}{PROMPT_ACTION}"}]
_prompt_text = tokenizer.apply_chat_template(_msgs_prompt, add_generation_prompt=True, tokenize=False)
CACHED_PROMPT_IDS = tokenizer(_prompt_text, return_tensors="pt").input_ids
PROMPT_LEN = CACHED_PROMPT_IDS.shape[1]


ACTION_TOKEN_IDS = {}
for name in ACTION_NAMES:
    ACTION_TOKEN_IDS[name] = tokenizer.encode(name, add_special_tokens=False)[0]

print(f"Prompt: {PROMPT_LEN} tok | Actions: {ACTION_TOKEN_IDS}")

_msgs_full = [
    {"role": "user", "content": f"{_img_str}{PROMPT_ACTION}"},
    {"role": "assistant", "content": "forward"}
]
_full_text = tokenizer.apply_chat_template(_msgs_full, add_generation_prompt=False, tokenize=False)
_full_ids = tokenizer(_full_text, return_tensors="pt").input_ids[0]
print(f"Full: {len(_full_ids)} tok. Answer tokens:")
for i in range(PROMPT_LEN, len(_full_ids)):
    print(f"  pos {i}: '{tokenizer.decode([_full_ids[i].item()])}'")


def full_logits(model, input_ids, images, attention_mask=None):
    """Forward + ВСЕГДА lm_head."""
    images_tensor = model._process_images(images, input_ids.device)
    token_embd = model.decoder.token_embedding(input_ids)
    if images_tensor is not None:
        image_embd = model.vision_encoder(images_tensor)
        image_embd = model.MP(image_embd)
        token_embd = model._replace_img_tokens_with_embd(input_ids, token_embd, image_embd)
    hidden, _ = model.decoder(token_embd, attention_mask=attention_mask)
    return model.decoder.head(hidden)


@torch.no_grad()
def generate_action(model, pil_img):
    """Одношаговый: logits по 3 action-токенам."""
    input_ids = CACHED_PROMPT_IDS.to(device)
    images = [process_image(pil_img)]
    logits = full_logits(model, input_ids, images)
    last = logits[0, -1, :]
    scores = torch.tensor([last[ACTION_TOKEN_IDS[n]].item() for n in ACTION_NAMES])
    best = scores.argmax().item()
    return best, ACTION_NAMES[best]


def evaluate(model, env_id, num_episodes=5, max_steps=40):
    model.eval()
    successes = 0
    for ep in range(num_episodes):
        env = make_env(env_id)
        obs, _ = env.reset(seed=50000 + ep)
        for step in range(max_steps):
            pil = obs_to_pil(env.render())
            aid, _ = generate_action(model, pil)
            obs, r, term, trunc, _ = env.step(aid)
            if term or trunc:
                break
        if term and r > 0:
            successes += 1
        env.close()
    return {'sr': successes / num_episodes}


class SFTDataset(Dataset):
    def __init__(self, data):
        self.pixels, self.ids, self.labels, self.masks = [], [], [], []
        img_str = vlm_cfg.vlm_extra_tokens["image_token"] * vlm_cfg.mp_image_token_length
        for pil_img, action_name in data:
            self.pixels.append(process_image(pil_img))
            msgs = [
                {"role": "user", "content": f"{img_str}{PROMPT_ACTION}"},
                {"role": "assistant", "content": action_name}
            ]
            text = tokenizer.apply_chat_template(msgs, add_generation_prompt=False, tokenize=False)
            enc = tokenizer(text, return_tensors="pt", truncation=True,
                           max_length=vlm_cfg.lm_max_length, padding="max_length")
            ids = enc.input_ids.squeeze(0)
            mask = enc.attention_mask.squeeze(0)
            lab = ids.clone()
            lab[:PROMPT_LEN] = -100
            lab[mask == 0] = -100
            self.ids.append(ids); self.labels.append(lab); self.masks.append(mask)
        self.ids = torch.stack(self.ids)
        self.labels = torch.stack(self.labels)
        self.masks = torch.stack(self.masks)

    def __len__(self): return len(self.pixels)
    def __getitem__(self, i): return self.pixels[i], self.ids[i], self.labels[i], self.masks[i]


def sft_collate(batch):
    px, ids, lab, msk = zip(*batch)
    return list(px), torch.stack(ids), torch.stack(lab), torch.stack(msk)


print("\nTraining SFT...")
sft_model = VisionLanguageModel(vlm_cfg, load_backbone=True).to(device)
for p in sft_model.vision_encoder.parameters():
    p.requires_grad = False

train_ds = SFTDataset(train_data)
val_ds = SFTDataset(val_data)
train_dl = DataLoader(train_ds, batch_size=SFT_BS, shuffle=True, drop_last=True, collate_fn=sft_collate)
val_dl = DataLoader(val_ds, batch_size=SFT_BS, shuffle=False, collate_fn=sft_collate)

opt = torch.optim.AdamW([
    {'params': sft_model.MP.parameters(), 'lr': 5e-4},
    {'params': sft_model.decoder.parameters(), 'lr': 5e-5},
], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, SFT_EPOCHS)

sft_hist = {'train_loss': [], 'val_loss': [], 'eval_ep': [], 'sr6': [], 'sr8': [], 'sr16': []}
best_sr, best_sd = -1, None

for epoch in range(SFT_EPOCHS):
    t0 = time.time()
    sft_model.train(); sft_model.vision_encoder.eval()
    tl, tn = 0, 0
    for px, ids, lab, msk in train_dl:
        ids, lab, msk = ids.to(device), lab.to(device), msk.to(device)
        _, loss = sft_model(ids, px, attention_mask=msk, targets=lab)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(sft_model.parameters(), 1.0)
        opt.step()
        tl += loss.item() * ids.size(0); tn += ids.size(0)
    scheduler.step()

    sft_model.eval()
    vl, vn = 0, 0
    with torch.no_grad():
        for px, ids, lab, msk in val_dl:
            ids, lab, msk = ids.to(device), lab.to(device), msk.to(device)
            _, loss = sft_model(ids, px, attention_mask=msk, targets=lab)
            vl += loss.item() * ids.size(0); vn += ids.size(0)

    avg_tl = tl/max(tn,1); avg_vl = vl/max(vn,1)
    sft_hist['train_loss'].append(avg_tl); sft_hist['val_loss'].append(avg_vl)

    # Eval + verification
    aid, gen = generate_action(sft_model, train_data[0][0])
    r8 = evaluate(sft_model, 'MiniGrid-Empty-8x8-v0', SFT_EVAL_N, 40)
    sr8 = r8['sr']

    if epoch == 0 or epoch == SFT_EPOCHS-1 or sr8 > 0:
        r6 = evaluate(sft_model, 'MiniGrid-Empty-6x6-v0', SFT_EVAL_N, 25)
        r16 = evaluate(sft_model, 'MiniGrid-Empty-16x16-v0', SFT_EVAL_N, 60)
    else:
        r6 = {'sr': 0}; r16 = {'sr': 0}

    sft_hist['eval_ep'].append(epoch)
    sft_hist['sr6'].append(r6['sr']); sft_hist['sr8'].append(sr8); sft_hist['sr16'].append(r16['sr'])

    if sr8 > best_sr:
        best_sr = sr8; best_sd = copy.deepcopy(sft_model.state_dict())

    print(f"  Ep{epoch+1}/{SFT_EPOCHS} L:{avg_tl:.4f}/{avg_vl:.4f} gen='{gen}' "
          f"SR 6={r6['sr']:.2f} 8={sr8:.2f} 16={r16['sr']:.2f} [{time.time()-t0:.0f}s]")

if best_sd: sft_model.load_state_dict(best_sd)
torch.save(sft_model.state_dict(), "results/checkpoints/sft_baseline.pt")
print(f"Best 8x8 SR: {best_sr:.2f}")

# Verification
print("\nVerification (5 train samples):")
for i in range(5):
    pil, true = train_data[i]
    aid, gen = generate_action(sft_model, pil)
    logits = full_logits(sft_model, CACHED_PROMPT_IDS.to(device), [process_image(pil)])
    al = {n: logits[0,-1,ACTION_TOKEN_IDS[n]].item() for n in ACTION_NAMES}
    print(f"  T:{true:>8s} G:{gen:>8s} L={al['left']:.1f} R={al['right']:.1f} F={al['forward']:.1f}")

print("\nFinal eval:")
for eid, ms in [('MiniGrid-Empty-6x6-v0',25),('MiniGrid-Empty-8x8-v0',40),('MiniGrid-Empty-16x16-v0',60)]:
    r = evaluate(sft_model, eid, FINAL_EVAL_N, ms)
    print(f"  {eid.split('-')[2]}: SR={r['sr']:.3f}")

import shutil
shutil.copy("results/checkpoints/sft_baseline.pt", f"{SAVE_DIR}/checkpoints/sft_baseline.pt")

Prompt: 103 tok | Actions: {'left': 8842, 'right': 2771, 'forward': 10403}
Full: 106 tok. Answer tokens:
  pos 103: 'forward'
  pos 104: '<|im_end|>'
  pos 105: '
'

Training SFT...
Loading from backbone weights
Successfully loaded google/siglip2-base-patch16-512 weights from safetensors. Model has 86,433,024 parameters.
Extending token embeddings from torch.Size([49152, 576]) to torch.Size([49218, 576])
Initialized 66 new token embeddings
Successfully loaded HuggingFaceTB/SmolLM2-135M weights from safetensors. Model has 134,553,024 parameters.
  Ep1/6 L:0.3192/0.0010 gen='right' SR 6=0.00 8=0.00 16=0.00 [173s]
  Ep2/6 L:0.0006/0.0004 gen='right' SR 6=0.00 8=0.00 16=0.00 [120s]
  Ep3/6 L:0.0003/0.0002 gen='right' SR 6=0.00 8=0.00 16=0.00 [118s]
  Ep4/6 L:0.0002/0.0002 gen='right' SR 6=0.00 8=0.00 16=0.00 [117s]
  Ep5/6 L:0.0002/0.0001 gen='right' SR 6=0.00 8=0.00 16=0.00 [117s]
  Ep6/6 L:0.0001/0.0001 gen='right' SR 6=0.00 8=0.00 16=0.00 [163s]
Best 8x8 SR: 0.00

Verification (5 train 

'/content/drive/MyDrive/nanoVLM-MiniGrid/checkpoints/sft_baseline.pt'

GRPO: прямой вывод действий (обязательно).

- Инициализируем политику весами из SFT-бэйзлайна (п. 1).
- Реализуем цикл обучения GRPO в среде EmptyEnv , где модель напрямую выдаёт действие.
- Построим кривые обучения и сравните качество с SFT-бэйзлайном.

In [37]:
import io, gc, copy, time
import numpy as np
import torch.nn.functional as F

sft_sd = torch.load("results/checkpoints/sft_baseline.pt", map_location='cpu', weights_only=True)

def grpo_step(model, env_id, group_size, max_steps, temp, base_seed):
    model.eval()
    prompt_ids = CACHED_PROMPT_IDS.to(device)
    rets, all_data = [], []
    for g in range(group_size):
        env = make_env(env_id)
        obs, _ = env.reset(seed=base_seed+g)
        steps = []
        for s in range(max_steps):
            pil = obs_to_pil(env.render())
            images = [process_image(pil)]
            with torch.no_grad():
                logits = full_logits(model, prompt_ids, images)
            act_logits = torch.tensor([logits[0,-1,ACTION_TOKEN_IDS[n]].item()
                                       for n in ACTION_NAMES], device=device) / temp
            probs = F.softmax(act_logits, dim=-1)
            act_idx = torch.multinomial(probs, 1).item()
            buf = io.BytesIO(); pil.save(buf, format='PNG')
            steps.append((buf.getvalue(), act_idx))
            obs, r, term, trunc, _ = env.step(act_idx)
            if term or trunc: break
        env.close()
        rets.append(1.0 if (term and r > 0) else 0.0)
        all_data.append(steps)
    mean_r = np.mean(rets); std_r = max(np.std(rets), 1e-8)
    advs = [(r - mean_r)/std_r for r in rets]
    if std_r < 1e-6:
        return 0.0, mean_r, int(sum(rets))
    model.train(); model.vision_encoder.eval()
    loss = torch.tensor(0.0, device=device)
    n = 0
    for g in range(group_size):
        if abs(advs[g]) < 0.1: continue
        for png, act_idx in all_data[g][:10]:
            pil = Image.open(io.BytesIO(png))
            images = [process_image(pil)]
            logits = full_logits(model, prompt_ids, images)
            act_logits = torch.stack([logits[0,-1,ACTION_TOKEN_IDS[nm]]
                                      for nm in ACTION_NAMES]) / temp
            log_probs = F.log_softmax(act_logits, dim=-1)
            loss = loss - log_probs[act_idx] * advs[g]
            n += 1
    if n > 0: loss = loss / n
    return loss, mean_r, int(sum(rets))

print("=" * 50)
print("GRPO: Direct Action")
print("=" * 50)

policy = VisionLanguageModel(vlm_cfg, load_backbone=False)
policy.load_state_dict(sft_sd); policy = policy.to(device)
for p in policy.vision_encoder.parameters(): p.requires_grad = False

opt = torch.optim.AdamW([
    {'params': policy.MP.parameters(), 'lr': 1e-5},
    {'params': policy.decoder.parameters(), 'lr': 1e-5},
], weight_decay=1e-4)

grpo_act_hist = {'loss':[], 'sr':[], 'eval_it':[], 'sr6':[], 'sr8':[], 'sr16':[]}

for it in range(GRPO_ITERS):
    t0 = time.time()
    loss, avg_r, ns = grpo_step(policy, 'MiniGrid-Empty-8x8-v0',
                                 GRPO_GROUP, 40, 0.5, it*1000)
    if isinstance(loss, torch.Tensor) and loss.requires_grad:
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0); opt.step()
        lv = loss.item()
    else: lv = 0.0
    torch.cuda.empty_cache()
    grpo_act_hist['loss'].append(lv); grpo_act_hist['sr'].append(avg_r)
    do_eval = (it+1) == GRPO_ITERS or (it+1) % 10 == 0
    if do_eval:
        r6 = evaluate(policy, 'MiniGrid-Empty-6x6-v0', GRPO_EVAL_N, 25)
        r8 = evaluate(policy, 'MiniGrid-Empty-8x8-v0', GRPO_EVAL_N, 40)
        r16 = evaluate(policy, 'MiniGrid-Empty-16x16-v0', GRPO_EVAL_N, 60)
        grpo_act_hist['eval_it'].append(it)
        grpo_act_hist['sr6'].append(r6['sr']); grpo_act_hist['sr8'].append(r8['sr']); grpo_act_hist['sr16'].append(r16['sr'])
        print(f"  [{it+1}/{GRPO_ITERS}] L={lv:.4f} SR={ns}/{GRPO_GROUP} "
              f"eval 6={r6['sr']:.2f} 8={r8['sr']:.2f} 16={r16['sr']:.2f} [{time.time()-t0:.0f}s]")
    else:
        print(f"  [{it+1}/{GRPO_ITERS}] L={lv:.4f} SR={ns}/{GRPO_GROUP} [{time.time()-t0:.0f}s]")

torch.save(policy.state_dict(), "results/checkpoints/grpo_action.pt")
grpo_action_model = policy

print("\nFinal eval GRPO-action:")
for eid, ms in [('MiniGrid-Empty-6x6-v0',25),('MiniGrid-Empty-8x8-v0',40),('MiniGrid-Empty-16x16-v0',60)]:
    r = evaluate(grpo_action_model, eid, FINAL_EVAL_N, ms)
    print(f"  {eid.split('-')[2]}: SR={r['sr']:.3f}")

torch.save(policy.state_dict(), "results/checkpoints/grpo_action.pt")
shutil.copy("results/checkpoints/grpo_action.pt", f"{SAVE_DIR}/checkpoints/grpo_action.pt")

GRPO: Direct Action
  [1/20] L=0.0000 SR=0/3 [16s]
  [2/20] L=0.0000 SR=0/3 [14s]
  [3/20] L=0.0000 SR=0/3 [14s]
  [4/20] L=0.0000 SR=0/3 [13s]
  [5/20] L=0.0000 SR=0/3 [13s]
  [6/20] L=0.0000 SR=0/3 [13s]
  [7/20] L=0.0000 SR=0/3 [13s]
  [8/20] L=0.0000 SR=0/3 [13s]
  [9/20] L=0.0000 SR=0/3 [13s]
  [10/20] L=0.0000 SR=0/3 eval 6=0.00 8=0.00 16=0.00 [81s]
  [11/20] L=0.0000 SR=0/3 [13s]
  [12/20] L=0.0000 SR=0/3 [13s]
  [13/20] L=0.0000 SR=0/3 [13s]
  [14/20] L=0.0000 SR=0/3 [13s]
  [15/20] L=0.0000 SR=0/3 [13s]
  [16/20] L=0.0000 SR=0/3 [13s]
  [17/20] L=0.0000 SR=0/3 [13s]
  [18/20] L=0.0000 SR=0/3 [13s]
  [19/20] L=0.0000 SR=0/3 [13s]
  [20/20] L=0.0000 SR=0/3 eval 6=0.00 8=0.00 16=0.00 [81s]

Final eval GRPO-action:
  6x6: SR=0.000
  8x8: SR=0.000
  16x16: SR=0.000


'/content/drive/MyDrive/nanoVLM-MiniGrid/checkpoints/grpo_action.pt'

GRPO: текст + действие (обязательно).

- Изменим формат вывода: модель генерирует краткое текстовое описание состояния (или «плана») и следующее действие. Выберите формат описания (2–3 предложения) и кратко обоснуйте его.
- Запустим GRPO-дообучение в этом формате.

In [39]:
### Cell 8: GRPO Text+Action

import io, gc, copy, time
import numpy as np
import torch.nn.functional as F
from PIL import Image

def parse_action(text):
    t = text.lower().strip()
    if t in NAME_TO_ACTION:
        return NAME_TO_ACTION[t]
    for name in ["forward", "right", "left"]:
        if name in t:
            return NAME_TO_ACTION[name]
    return 2  # default: forward

# Загружаем SFT веса
sft_sd = torch.load("results/checkpoints/sft_baseline.pt", map_location='cpu', weights_only=True)
gc.collect(); torch.cuda.empty_cache()

# CoT промпт
PROMPT_COT = (
    "You control an agent in a grid world. "
    "It must reach the green goal. "
    "Briefly describe what you see, then choose: left, right, or forward."
)
_img_str = vlm_cfg.vlm_extra_tokens["image_token"] * vlm_cfg.mp_image_token_length
_cot_msgs = [{"role": "user", "content": f"{_img_str}{PROMPT_COT}"}]
_cot_text = tokenizer.apply_chat_template(_cot_msgs, add_generation_prompt=True, tokenize=False)
CACHED_COT_IDS = tokenizer(_cot_text, return_tensors="pt").input_ids
COT_PROMPT_LEN = CACHED_COT_IDS.shape[1]
print(f"CoT prompt: {COT_PROMPT_LEN} tok")

# ID image-токена для фильтрации
IMG_TOKEN_ID = tokenizer.convert_tokens_to_ids('<|image|>')

def grpo_text_step(model, env_id, group_size, max_steps, temp, max_gen, base_seed):
    model.eval()
    prompt_ids = CACHED_COT_IDS.to(device)
    rets, all_data = [], []

    for g in range(group_size):
        env = make_env(env_id)
        obs, _ = env.reset(seed=base_seed + g)
        steps = []
        for s in range(max_steps):
            pil = obs_to_pil(env.render())
            images = [process_image(pil)]

            # Авторегрессивная генерация
            gen_ids = prompt_ids.clone()
            gen_toks = []

            for t in range(max_gen):
                with torch.no_grad():
                    # images только на первом шаге, потом None
                    logits = full_logits(model, gen_ids, images if t == 0 else None)

                next_logits = logits[0, -1, :] / temp
                # Запрещаем генерировать image-токен
                next_logits[IMG_TOKEN_ID] = -float('inf')
                probs = F.softmax(next_logits, dim=-1)
                tok = torch.multinomial(probs, 1)  # [1]
                gen_toks.append(tok.item())
                gen_ids = torch.cat([gen_ids, tok.unsqueeze(0)], dim=1)

                if tok.item() == tokenizer.eos_token_id:
                    break

            text = tokenizer.decode(gen_toks, skip_special_tokens=True)
            aid = parse_action(text)
            buf = io.BytesIO(); pil.save(buf, format='PNG')
            steps.append((buf.getvalue(), gen_toks, aid))

            obs, r, term, trunc, _ = env.step(aid)
            if term or trunc:
                break
        env.close()
        rets.append(1.0 if (term and r > 0) else 0.0)
        all_data.append(steps)

    mean_r = np.mean(rets); std_r = max(np.std(rets), 1e-8)
    advs = [(r - mean_r) / std_r for r in rets]
    if std_r < 1e-6:
        return 0.0, mean_r, int(sum(rets))

    # Backward pass
    model.train(); model.vision_encoder.eval()
    loss = torch.tensor(0.0, device=device)
    n = 0
    for g in range(group_size):
        if abs(advs[g]) < 0.1:
            continue
        for png, gen_toks_list, aid in all_data[g][:10]:
            pil = Image.open(io.BytesIO(png))
            images = [process_image(pil)]
            # Градиент только по промпту → logits последней позиции → 3 action-токена
            logits = full_logits(model, prompt_ids, images)
            act_logits = torch.stack([logits[0, -1, ACTION_TOKEN_IDS[nm]]
                                      for nm in ACTION_NAMES]) / temp
            log_probs = F.log_softmax(act_logits, dim=-1)
            loss = loss - log_probs[aid] * advs[g]
            n += 1
    if n > 0:
        loss = loss / n
    return loss, mean_r, int(sum(rets))

print("=" * 50)
print("GRPO: Text+Action")
print("=" * 50)

policy2 = VisionLanguageModel(vlm_cfg, load_backbone=False)
policy2.load_state_dict(sft_sd)
policy2 = policy2.to(device)
for p in policy2.vision_encoder.parameters():
    p.requires_grad = False

opt2 = torch.optim.AdamW([
    {'params': policy2.MP.parameters(), 'lr': 1e-5},
    {'params': policy2.decoder.parameters(), 'lr': 1e-5},
], weight_decay=1e-4)

grpo_text_hist = {'loss': [], 'sr': [], 'eval_it': [], 'sr6': [], 'sr8': [], 'sr16': []}

for it in range(GRPO_ITERS):
    t0 = time.time()
    loss, avg_r, ns = grpo_text_step(policy2, 'MiniGrid-Empty-8x8-v0',
                                      GRPO_GROUP, 40, 0.5, 12, it * 1000 + 50000)
    if isinstance(loss, torch.Tensor) and loss.requires_grad:
        opt2.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy2.parameters(), 1.0)
        opt2.step()
        lv = loss.item()
    else:
        lv = 0.0
    torch.cuda.empty_cache()
    grpo_text_hist['loss'].append(lv)
    grpo_text_hist['sr'].append(avg_r)

    do_eval = (it + 1) == GRPO_ITERS or (it + 1) % 10 == 0
    if do_eval:
        r6 = evaluate(policy2, 'MiniGrid-Empty-6x6-v0', GRPO_EVAL_N, 25)
        r8 = evaluate(policy2, 'MiniGrid-Empty-8x8-v0', GRPO_EVAL_N, 40)
        r16 = evaluate(policy2, 'MiniGrid-Empty-16x16-v0', GRPO_EVAL_N, 60)
        grpo_text_hist['eval_it'].append(it)
        grpo_text_hist['sr6'].append(r6['sr'])
        grpo_text_hist['sr8'].append(r8['sr'])
        grpo_text_hist['sr16'].append(r16['sr'])
        print(f"  [{it+1}/{GRPO_ITERS}] L={lv:.4f} SR={ns}/{GRPO_GROUP} "
              f"eval 6={r6['sr']:.2f} 8={r8['sr']:.2f} 16={r16['sr']:.2f} [{time.time()-t0:.0f}s]")
    else:
        print(f"  [{it+1}/{GRPO_ITERS}] L={lv:.4f} SR={ns}/{GRPO_GROUP} [{time.time()-t0:.0f}s]")

torch.save(policy2.state_dict(), "results/checkpoints/grpo_text.pt")
import shutil
shutil.copy("results/checkpoints/grpo_text.pt", f"{SAVE_DIR}/checkpoints/grpo_text.pt")
grpo_text_model = policy2

print("\nSample generated texts:")
policy2.eval()
sample_env = make_env('MiniGrid-Empty-8x8-v0')
for ep in range(3):
    obs, _ = sample_env.reset(seed=99000 + ep)
    pil = obs_to_pil(sample_env.render())
    images = [process_image(pil)]
    gen_ids = CACHED_COT_IDS.to(device).clone()
    gen_toks = []
    for t in range(12):
        with torch.no_grad():
            logits = full_logits(policy2, gen_ids, images if t == 0 else None)
        next_logits = logits[0, -1, :] / 0.5
        next_logits[IMG_TOKEN_ID] = -float('inf')
        probs = F.softmax(next_logits, dim=-1)
        tok = torch.multinomial(probs, 1)
        gen_toks.append(tok.item())
        gen_ids = torch.cat([gen_ids, tok.unsqueeze(0)], dim=1)
        if tok.item() == tokenizer.eos_token_id:
            break
    text = tokenizer.decode(gen_toks, skip_special_tokens=True)
    print(f"  Ep{ep}: '{text}' → action={parse_action(text)}")
sample_env.close()

print("\nFinal eval GRPO-text:")
for eid, ms in [('MiniGrid-Empty-6x6-v0', 25), ('MiniGrid-Empty-8x8-v0', 40), ('MiniGrid-Empty-16x16-v0', 60)]:
    r = evaluate(grpo_text_model, eid, FINAL_EVAL_N, ms)
    print(f"  {eid.split('-')[2]}: SR={r['sr']:.3f}")

CoT prompt: 106 tok
GRPO: Text+Action
  [1/20] L=0.0000 SR=0/3 [57s]
  [2/20] L=0.7802 SR=1/3 [52s]
  [3/20] L=0.0000 SR=0/3 [57s]
  [4/20] L=0.0000 SR=3/3 [22s]
  [5/20] L=0.0000 SR=0/3 [56s]
  [6/20] L=-0.5079 SR=1/3 [57s]
  [7/20] L=0.0000 SR=1/3 [61s]
  [8/20] L=1.3466 SR=1/3 [50s]
  [9/20] L=-0.6455 SR=1/3 [53s]
  [10/20] L=0.0000 SR=3/3 eval 6=0.00 8=0.00 16=0.00 [106s]
  [11/20] L=0.0000 SR=0/3 [56s]
  [12/20] L=0.0000 SR=2/3 [45s]
  [13/20] L=0.7364 SR=2/3 [48s]
  [14/20] L=0.0000 SR=0/3 [57s]
  [15/20] L=0.0000 SR=0/3 [56s]
  [16/20] L=0.0000 SR=0/3 [57s]
  [17/20] L=0.6729 SR=2/3 [40s]
  [18/20] L=0.5332 SR=1/3 [50s]
  [19/20] L=0.0000 SR=0/3 [57s]
  [20/20] L=-1.2814 SR=2/3 eval 6=0.00 8=0.00 16=0.00 [118s]

Sample generated texts:
  Ep0: '
What does the word "forever" mean?
' → action=2
  Ep1: '
The right hand is the right hand.

The' → action=1
  Ep2: '
Zoe's story reminds me of a time when I' → action=2

Final eval GRPO-text:
  6x6: SR=0.000
  8x8: SR=0.000
  16x16: SR=0.

Сравнение

In [41]:
### Cell 9: Графики и сравнение (быстрая версия)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import shutil

# ====== График 1: SFT ======
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(sft_hist['train_loss'], label='Train', lw=2)
axes[0].plot(sft_hist['val_loss'], label='Val', lw=2)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('SFT Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(sft_hist['eval_ep'], sft_hist['sr6'], 'o-', label='6×6')
axes[1].plot(sft_hist['eval_ep'], sft_hist['sr8'], 's-', label='8×8')
axes[1].plot(sft_hist['eval_ep'], sft_hist['sr16'], '^-', label='16×16')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Success Rate')
axes[1].set_title('SFT Eval'); axes[1].legend()
axes[1].set_ylim(-0.05, 1.05); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("results/plots/sft_baseline.png", dpi=150, bbox_inches='tight')
shutil.copy("results/plots/sft_baseline.png", f"{SAVE_DIR}/plots/sft_baseline.png")
plt.show()
print("✓ SFT plot saved")

# ====== График 2: GRPO оба метода ======
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Train SR
for h, lb, c in [(grpo_act_hist, 'GRPO-Action', '#FF9800'),
                  (grpo_text_hist, 'GRPO-Text', '#4CAF50')]:
    axes[0].plot(h['sr'], alpha=0.3, color=c)
    ww = 3
    if len(h['sr']) >= ww:
        sm = np.convolve(h['sr'], np.ones(ww)/ww, 'valid')
        axes[0].plot(range(ww-1, len(h['sr'])), sm, lw=2, label=lb, color=c)
    else:
        axes[0].plot(h['sr'], lw=2, label=lb, color=c)
axes[0].set_xlabel('Iter'); axes[0].set_ylabel('Success Rate')
axes[0].set_title('GRPO Train SR'); axes[0].legend()
axes[0].set_ylim(-0.05, 1.05); axes[0].grid(True, alpha=0.3)

# Loss
for h, lb, c in [(grpo_act_hist, 'GRPO-Action', '#FF9800'),
                  (grpo_text_hist, 'GRPO-Text', '#4CAF50')]:
    axes[1].plot(h['loss'], alpha=0.7, label=lb, color=c)
axes[1].set_xlabel('Iter'); axes[1].set_ylabel('Loss')
axes[1].set_title('GRPO Loss'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("results/plots/grpo_comparison.png", dpi=150, bbox_inches='tight')
shutil.copy("results/plots/grpo_comparison.png", f"{SAVE_DIR}/plots/grpo_comparison.png")
plt.show()
print("✓ GRPO plot saved")

# ====== График 3: Финальное сравнение (bar chart) ======
sft_final = [sft_hist['sr6'][-1], sft_hist['sr8'][-1], sft_hist['sr16'][-1]]
act_final = [grpo_act_hist['sr6'][-1], grpo_act_hist['sr8'][-1], grpo_act_hist['sr16'][-1]]
txt_final = [grpo_text_hist['sr6'][-1], grpo_text_hist['sr8'][-1], grpo_text_hist['sr16'][-1]]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(3)
w = 0.25
ax.bar(x - w, sft_final, w, label='SFT', color='#2196F3')
ax.bar(x,     act_final, w, label='GRPO-Action', color='#FF9800')
ax.bar(x + w, txt_final, w, label='GRPO-Text', color='#4CAF50')
ax.set_xticks(x)
ax.set_xticklabels(['6×6', '8×8', '16×16'])
ax.set_ylabel('Success Rate')
ax.set_title('Final Comparison: SFT vs GRPO-Action vs GRPO-Text')
ax.legend()
ax.set_ylim(0, 1.15)
ax.grid(True, alpha=0.3, axis='y')

for i in range(3):
    ax.text(i - w, sft_final[i] + 0.02, f'{sft_final[i]:.2f}', ha='center', fontsize=9)
    ax.text(i,     act_final[i] + 0.02, f'{act_final[i]:.2f}', ha='center', fontsize=9)
    ax.text(i + w, txt_final[i] + 0.02, f'{txt_final[i]:.2f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig("results/plots/final_comparison.png", dpi=150, bbox_inches='tight')
shutil.copy("results/plots/final_comparison.png", f"{SAVE_DIR}/plots/final_comparison.png")
plt.show()

# ====== Итоговая таблица ======
print("\n" + "=" * 50)
print("FINAL RESULTS")
print("=" * 50)
print(f"{'Method':<15} {'6×6':>6} {'8×8':>6} {'16×16':>7}")
print("-" * 35)
print(f"{'SFT':<15} {sft_final[0]:>6.3f} {sft_final[1]:>6.3f} {sft_final[2]:>7.3f}")
print(f"{'GRPO-Action':<15} {act_final[0]:>6.3f} {act_final[1]:>6.3f} {act_final[2]:>7.3f}")
print(f"{'GRPO-Text':<15} {txt_final[0]:>6.3f} {txt_final[1]:>6.3f} {txt_final[2]:>7.3f}")
print(f"\n✓ All plots saved to {SAVE_DIR}/plots/")

✓ SFT plot saved
✓ GRPO plot saved

FINAL RESULTS
Method             6×6    8×8   16×16
-----------------------------------
SFT              0.000  0.000   0.000
GRPO-Action      0.000  0.000   0.000
GRPO-Text        0.000  0.000   0.000

✓ All plots saved to /content/drive/MyDrive/nanoVLM-MiniGrid/plots/
